# Section 1 — literature retrieval, topic modelling, and ontology-guided extraction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/romenmeitei/AISKG_01_Framework/blob/main/Mushroom_KG_Upstream_Literature_to_Extraction_Pipeline_v1.ipynb)

This notebook is the **upstream first section** of the reproducibility repository. It precedes the canonical post-extraction workflow.

- **`MANUSCRIPT_SNAPSHOT` (default, tested):** verifies the frozen corpus and expert-curation inputs, replays topic consolidation, regenerates the ontology-guided entity and relation tables, audits them against the frozen study outputs, and creates a complete input ZIP for Section 2.
- **`LIVE_REFRESH` (optional):** runs the recorded PubMed search and, when licensed API credentials are available, Scopus and Web of Science retrieval; performs harmonization and deduplication; fits `all-mpnet-base-v2` + UMAP + HDBSCAN + BERTopic; and creates an expert-review template. A second live phase applies the completed topic review and regenerates semantic extraction.

The frozen snapshot is the authoritative route for the manuscript numbers. A live refresh is expected to differ because bibliographic databases and software models change over time.

In [1]:
# ================================ CONFIGURATION ================================
RUN_MODE = "MANUSCRIPT_SNAPSHOT"   # "MANUSCRIPT_SNAPSHOT" or "LIVE_REFRESH"
LIVE_PHASE = "RETRIEVE_AND_MODEL"  # or "APPLY_CURATION_AND_EXTRACT"
LIVE_EXPERT_REVIEW_FILE = ""       # path to completed live topic-review CSV
FALLBACK_TO_SNAPSHOT = True         # use exact snapshot if a live service fails
AUTO_DOWNLOAD_OUTPUTS = True

INPUT_ZIP_NAME = "Mushroom_KG_Upstream_Inputs_v1.zip"
OUTPUT_ZIP_NAME = "Mushroom_KG_Upstream_Reproducibility_Outputs.zip"
BRIDGE_ZIP_NAME = "Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip"

print("Run mode:", RUN_MODE)
if RUN_MODE.upper() == "LIVE_REFRESH":
    print("Live phase:", LIVE_PHASE)

Run mode: MANUSCRIPT_SNAPSHOT


In [2]:
# Install only packages that are missing. The default snapshot mode does not
# download language models or contact external databases.
import importlib.util
import subprocess
import sys

snapshot_requirements = {
    "pandas": "pandas>=2.2,<3",
    "numpy": "numpy>=1.26,<3",
    "scipy": "scipy>=1.13,<2",
    "spacy": "spacy>=3.8,<3.9",
    "matplotlib": "matplotlib>=3.8,<4",
    "openpyxl": "openpyxl>=3.1,<4",
}
live_requirements = {
    "requests": "requests>=2.32,<3",
    "tqdm": "tqdm>=4.66,<5",
    "Bio": "biopython>=1.84,<2",
    "rapidfuzz": "rapidfuzz>=3.9,<4",
    "bertopic": "bertopic>=0.17,<0.18",
    "sentence_transformers": "sentence-transformers>=5,<6",
    "umap": "umap-learn>=0.5,<0.6",
    "hdbscan": "hdbscan>=0.8,<0.9",
}
requirements = dict(snapshot_requirements)
if RUN_MODE.upper() == "LIVE_REFRESH":
    requirements.update(live_requirements)

missing = [spec for module, spec in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")

All required packages are available.


In [3]:
# Locate/download/upload the companion input ZIP, extract it, and load the
# versioned pipeline module stored inside the checksummed bundle.
from pathlib import Path
import importlib.util
import os
import shutil
import sys
import urllib.request
import zipfile

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

candidates = []
override = os.environ.get("MUSHROOM_UPSTREAM_INPUT_ZIP", "").strip()
if override:
    candidates.append(Path(override))
candidates.extend([Path.cwd() / INPUT_ZIP_NAME, Path("/content") / INPUT_ZIP_NAME])
input_zip = next((path for path in candidates if path.exists()), None)

if input_zip is None:
    raw_url = (
        "https://raw.githubusercontent.com/romenmeitei/"
        "AISKG_01_Framework/"
        "main/Mushroom_KG_Upstream_Inputs_v1.zip"
    )
    try:
        print("Trying repository companion bundle:", raw_url)
        urllib.request.urlretrieve(raw_url, INPUT_ZIP_NAME)
        input_zip = Path(INPUT_ZIP_NAME)
    except Exception as download_error:
        print("Automatic repository download was unavailable:", download_error)
        if IN_COLAB:
            print(f"Upload {INPUT_ZIP_NAME}")
            uploaded = colab_files.upload()
            if INPUT_ZIP_NAME in uploaded:
                Path(INPUT_ZIP_NAME).write_bytes(uploaded[INPUT_ZIP_NAME])
                input_zip = Path(INPUT_ZIP_NAME)
            elif uploaded:
                name, content = next(iter(uploaded.items()))
                input_zip = Path(name).name
                Path(input_zip).write_bytes(content)
        if input_zip is None:
            raise FileNotFoundError(
                f"Could not locate {INPUT_ZIP_NAME}. Place it beside the notebook or upload it in Colab."
            ) from download_error

WORK_ROOT = Path("/content/mushroom_kg_upstream_v1") if IN_COLAB else Path.cwd() / "mushroom_kg_upstream_run_v1"
INPUT_EXTRACT_ROOT = WORK_ROOT / "inputs"
OUTPUT_ROOT = WORK_ROOT / "outputs"
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
INPUT_EXTRACT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(input_zip, "r") as archive:
    archive.extractall(INPUT_EXTRACT_ROOT)

manifests = sorted(INPUT_EXTRACT_ROOT.rglob("input_checksums.csv"), key=lambda p: (len(p.parts), str(p)))
root_manifests = [p for p in manifests if p.parent == INPUT_EXTRACT_ROOT]
if len(root_manifests) != 1:
    raise RuntimeError(f"Expected exactly one root input_checksums.csv; found {len(root_manifests)}")
INPUT_ROOT = root_manifests[0].parent
CORE_PATH = INPUT_ROOT / "code" / "upstream_core.py"
if not CORE_PATH.exists():
    raise FileNotFoundError(f"Missing pipeline module: {CORE_PATH}")

spec = importlib.util.spec_from_file_location("mushroom_upstream_core", CORE_PATH)
core = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = core
spec.loader.exec_module(core)

print("Input ZIP:", Path(input_zip).resolve())
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Pipeline version:", core.PIPELINE_VERSION)

Input ZIP: /mnt/data/repo_release_build/exec_AISKG_01_Framework_UPLOAD_READY/Mushroom_KG_Upstream_Inputs_v1.zip
Input root: /mnt/data/repo_release_build/exec_AISKG_01_Framework_UPLOAD_READY/mushroom_kg_upstream_run_v1/inputs
Output root: /mnt/data/repo_release_build/exec_AISKG_01_Framework_UPLOAD_READY/mushroom_kg_upstream_run_v1/outputs
Pipeline version: 1.0.0


In [4]:
# Execute the selected workflow. The default route is a complete offline replay.
import json
import os
from pathlib import Path


def read_secret(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception:
        return ""

mode = RUN_MODE.upper().strip()
if mode == "MANUSCRIPT_SNAPSHOT":
    result = core.run_snapshot_pipeline(INPUT_ROOT, OUTPUT_ROOT)
    EFFECTIVE_MODE = mode
elif mode == "LIVE_REFRESH":
    config = core.LiveConfig()
    review_path = Path(LIVE_EXPERT_REVIEW_FILE).expanduser() if LIVE_EXPERT_REVIEW_FILE else None
    try:
        result = core.run_live_refresh(
            OUTPUT_ROOT,
            phase=LIVE_PHASE,
            config=config,
            entrez_email=read_secret("ENTREZ_EMAIL"),
            entrez_api_key=read_secret("NCBI_API_KEY"),
            scopus_api_key=read_secret("SCOPUS_API_KEY"),
            wos_api_key=read_secret("WOS_API_KEY"),
            annotated_topic_file=review_path,
            ontology_aliases=INPUT_ROOT / "data" / "ontology" / "ontology_entity_aliases.csv",
            relation_rules_file=INPUT_ROOT / "data" / "ontology" / "relation_rules.json",
        )
        live_archive = WORK_ROOT / OUTPUT_ZIP_NAME
        core.deterministic_zip(OUTPUT_ROOT, live_archive)
        result["output_archive"] = str(live_archive)
        (OUTPUT_ROOT / "LIVE_REFRESH_STAGE_SUCCESS.txt").write_text(
            f"LIVE_REFRESH {LIVE_PHASE} completed successfully.\n", encoding="utf-8"
        )
        EFFECTIVE_MODE = mode
    except Exception as live_error:
        if not FALLBACK_TO_SNAPSHOT:
            raise
        print("Live refresh did not complete:", repr(live_error))
        print("FALLBACK_TO_SNAPSHOT=True; running the frozen manuscript replay instead.")
        if OUTPUT_ROOT.exists():
            import shutil
            shutil.rmtree(OUTPUT_ROOT)
        result = core.run_snapshot_pipeline(INPUT_ROOT, OUTPUT_ROOT)
        result["live_refresh_error"] = repr(live_error)
        EFFECTIVE_MODE = "MANUSCRIPT_SNAPSHOT_FALLBACK"
else:
    raise ValueError("RUN_MODE must be MANUSCRIPT_SNAPSHOT or LIVE_REFRESH")

print("Effective mode:", EFFECTIVE_MODE)
print(json.dumps(result.get("run_manifest", result), indent=2, default=str))

Effective mode: MANUSCRIPT_SNAPSHOT
{
  "pipeline_name": "Mushroom KG upstream literature-to-extraction pipeline",
  "pipeline_version": "1.0.0",
  "mode": "MANUSCRIPT_SNAPSHOT",
  "generated_at_utc": "2026-08-01T12:13:22.361940+00:00",
  "python_version": "3.13.5",
  "platform": "Linux-6.12.13-x86_64-with-glibc2.41",
  "packages": {
    "pandas": "2.2.3",
    "numpy": "2.3.5",
    "spacy": "3.8.11",
    "scipy": "1.17.0",
    "matplotlib": "3.10.8"
  },
  "summary": {
    "full_corpus_records": 2687,
    "included_records": 1868,
    "curated_themes": 9,
    "topic_ids_reviewed": 56,
    "entity_mentions": 8292,
    "unique_entities": 92,
    "documents_with_entities": 1521,
    "explicit_relation_instances": 1324,
    "aggregated_semantic_edges": 183,
    "support_ge_2_edges": 86,
    "support_ge_2_active_nodes": 40,
    "source_pubmed_records": 1095,
    "source_scopus_records": 1592,
    "year_start": 2000,
    "year_end": 2025
  },
  "reference_files_passed": 11,
  "reference_file

In [5]:
# Show the audit results and verify the bridge handed to Section 2.
from pathlib import Path
import pandas as pd
import zipfile

if EFFECTIVE_MODE.startswith("MANUSCRIPT_SNAPSHOT"):
    audit_files = [
        ("Input checksums", OUTPUT_ROOT / "audits" / "01_input_checksum_audit.csv"),
        ("Frozen output comparison", OUTPUT_ROOT / "audits" / "02_reference_output_audit.csv"),
        ("Fixed numerical checks", OUTPUT_ROOT / "audits" / "03_fixed_result_checks.csv"),
        ("Section 2 bridge checks", OUTPUT_ROOT / "audits" / "04_post_extraction_bridge_audit.csv"),
    ]
    for title, path in audit_files:
        frame = pd.read_csv(path)
        print(f"\n{title}: {len(frame)} rows")
        if "status" in frame.columns:
            print(frame["status"].value_counts(dropna=False).to_dict())
        try:
            display(frame)
        except NameError:
            print(frame.head())

    success_path = OUTPUT_ROOT / "PIPELINE_SUCCESS.txt"
    if not success_path.exists():
        raise RuntimeError("The snapshot run did not create PIPELINE_SUCCESS.txt")
    print("\n", success_path.read_text().strip())

    bridge_zip = OUTPUT_ROOT / BRIDGE_ZIP_NAME
    if not bridge_zip.exists():
        raise FileNotFoundError(f"Section 2 bridge ZIP was not created: {bridge_zip}")
    with zipfile.ZipFile(bridge_zip) as archive:
        bridge_names = sorted(name for name in archive.namelist() if not name.endswith("/"))
    print(f"Section 2 bridge: {bridge_zip.name} ({len(bridge_names)} files)")
    print("Core regenerated files:")
    for name in bridge_names:
        if any(name.endswith(core_name) for core_name in core.KEY_POST_EXTRACTION_FILES):
            print(" -", name)
else:
    print("Live-stage outputs:")
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if path.is_file():
            print(" -", path.relative_to(OUTPUT_ROOT))


Input checksums: 39 rows
{'PASS': 39}


,file,expected_sha256,actual_sha256,expected_bytes,actual_bytes,status
0,code/upstream_core.py,7db1fca6a5648d8e7bffa3df7dc7c06b92f41407e88b89...,7db1fca6a5648d8e7bffa3df7dc7c06b92f41407e88b89...,80636,80636,PASS
1,config/search_queries_and_parameters.json,4880deb119d53760d8946ea6f432bc1f2bc865f84ad5d1...,4880deb119d53760d8946ea6f432bc1f2bc865f84ad5d1...,2480,2480,PASS
2,data/bridge_static/Blinded_Annotation_Expert_A...,ddbbe3d2300cb3a9adefcccd2a40364e641f9dfda2ae44...,ddbbe3d2300cb3a9adefcccd2a40364e641f9dfda2ae44...,103703,103703,PASS
3,data/bridge_static/Blinded_Annotation_Expert_B...,b459f252cdeda51a4a00042bb277ff66f65a51b5032035...,b459f252cdeda51a4a00042bb277ff66f65a51b5032035...,107203,107203,PASS
4,data/bridge_static/Table_validated_relation_in...,077587404168ac1fb3a6a8dedb5b8d9e3bfa8bc77bc25b...,077587404168ac1fb3a6a8dedb5b8d9e3bfa8bc77bc25b...,385322,385322,PASS
5,data/bridge_static/Table_validated_toxicity_pa...,950fa2f5467b07bfb4dfe70a837f6d8800692a3afe0921...,950fa2f5467b07bfb4dfe70a837f6d8800692a3afe0921...,16089,16089,PASS
6,data/bridge_static/Third_Expert_Adjudication_E...,6568b83b8ab0ba42cb2479441e27b7f85c50dea8a18320...,6568b83b8ab0ba42cb2479441e27b7f85c50dea8a18320...,27494,27494,PASS
7,data/bridge_static/data_dictionary.csv,248a42a41c36ff347dce5a372ffeac8ba8a0bc961da063...,248a42a41c36ff347dce5a372ffeac8ba8a0bc961da063...,5519,5519,PASS
8,data/bridge_static/frozen_scispacy_entity_reco...,3884af1d1eb8cd36d7b7495964e9d2950ad484264bb63f...,3884af1d1eb8cd36d7b7495964e9d2950ad484264bb63f...,29488,29488,PASS
9,data/bridge_static/heldout_gold_entities.csv,58e5b60086ba691e3a5ae8ce5ffa8b70230308fe0a6dc0...,58e5b60086ba691e3a5ae8ce5ffa8b70230308fe0a6dc0...,38139,38139,PASS



Frozen output comparison: 11 rows
{'PASS': 11}


,file,generated_sha256,reference_sha256,byte_identical,generated_bytes,reference_bytes,generated_rows,reference_rows,generated_columns,reference_columns,same_columns,cell_identical,cell_mismatches,status
0,03_full_corpus_with_final_themes.csv,e6ba97bfd37f3859864c1fcbfde7ef83e00afe1e188b80...,e6ba97bfd37f3859864c1fcbfde7ef83e00afe1e188b80...,True,5011284,5011284,2687,2687,23,23,True,True,0,PASS
1,04_included_corpus_final_themes_only.csv,b826b3795067394f310ab28dda7d91775574129f9c992f...,b826b3795067394f310ab28dda7d91775574129f9c992f...,True,3374941,3374941,1868,1868,23,23,True,True,0,PASS
2,05_final_theme_counts.csv,5da689a5342a31a8aa4eae2f9029dc0dc2d9d8a4356184...,5da689a5342a31a8aa4eae2f9029dc0dc2d9d8a4356184...,True,532,532,9,9,3,3,True,True,0,PASS
3,06_final_theme_temporal_trends_long.csv,39ba16a9cf588d0022a0eb69e74914413092361c5d4fa1...,39ba16a9cf588d0022a0eb69e74914413092361c5d4fa1...,True,10469,10469,195,195,3,3,True,True,0,PASS
4,07_final_theme_temporal_trends_wide.csv,6fbaf3cf2031f17132d4831d25b3e059e0f84060f7cf26...,6fbaf3cf2031f17132d4831d25b3e059e0f84060f7cf26...,True,1583,1583,26,26,10,10,True,True,0,PASS
5,09_final_theme_relative_contribution.csv,2df996ab3c7ce0aacaaccd80f1e03c418ad322f7c60265...,2df996ab3c7ce0aacaaccd80f1e03c418ad322f7c60265...,True,4173,4173,26,26,10,10,True,True,0,PASS
6,01_document_sentence_entities.csv,fa09e9d3a1c983f59b5f612986c15a648c62137b36f938...,fa09e9d3a1c983f59b5f612986c15a648c62137b36f938...,True,3341317,3341317,8292,8292,14,14,True,True,0,PASS
7,04_explicit_sentence_relations.csv,d9640cc309ab825ccfceee0af6803407165bf55bda3f93...,d9640cc309ab825ccfceee0af6803407165bf55bda3f93...,True,423835,423835,1324,1324,14,14,True,True,0,PASS
8,06_explicit_edges_aggregated.csv,59a4c89a19025e86e0b9c58751a20d6195540c4bf0d8cf...,59a4c89a19025e86e0b9c58751a20d6195540c4bf0d8cf...,True,47858,47858,183,183,14,14,True,True,0,PASS
9,13_entity_year_counts.csv,5771134a91e32e886aaf805f96ff7a4e78de5d566fc62a...,5771134a91e32e886aaf805f96ff7a4e78de5d566fc62a...,True,34160,34160,1088,1088,4,4,True,True,0,PASS



Fixed numerical checks: 15 rows
{'PASS': 15}


,metric,expected,actual,status
0,full_corpus_records,2687,2687,PASS
1,included_records,1868,1868,PASS
2,curated_themes,9,9,PASS
3,topic_ids_reviewed,56,56,PASS
4,entity_mentions,8292,8292,PASS
5,unique_entities,92,92,PASS
6,documents_with_entities,1521,1521,PASS
7,explicit_relation_instances,1324,1324,PASS
8,aggregated_semantic_edges,183,183,PASS
9,support_ge_2_edges,86,86,PASS



Section 2 bridge checks: 22 rows
{'PASS': 22}


,file,source,sha256,bytes,matches_frozen_manifest,status
0,01_document_sentence_entities.csv,generated_upstream,fa09e9d3a1c983f59b5f612986c15a648c62137b36f938...,3341317,True,PASS
1,03_full_corpus_with_final_themes.csv,generated_upstream,e6ba97bfd37f3859864c1fcbfde7ef83e00afe1e188b80...,5011284,True,PASS
2,04_explicit_sentence_relations.csv,generated_upstream,d9640cc309ab825ccfceee0af6803407165bf55bda3f93...,423835,True,PASS
3,06_explicit_edges_aggregated.csv,generated_upstream,59a4c89a19025e86e0b9c58751a20d6195540c4bf0d8cf...,47858,True,PASS
4,Blinded_Annotation_Expert_A.xlsx,frozen_static,ddbbe3d2300cb3a9adefcccd2a40364e641f9dfda2ae44...,103703,True,PASS
5,Blinded_Annotation_Expert_B.xlsx,frozen_static,b459f252cdeda51a4a00042bb277ff66f65a51b5032035...,107203,True,PASS
6,Table_validated_relation_instances.csv,frozen_static,077587404168ac1fb3a6a8dedb5b8d9e3bfa8bc77bc25b...,385322,True,PASS
7,Table_validated_toxicity_pathways_with_stabili...,frozen_static,950fa2f5467b07bfb4dfe70a837f6d8800692a3afe0921...,16089,True,PASS
8,Third_Expert_Adjudication_Expanded.xlsx,frozen_static,6568b83b8ab0ba42cb2479441e27b7f85c50dea8a18320...,27494,True,PASS
9,data_dictionary.csv,frozen_static,248a42a41c36ff347dce5a372ffeac8ba8a0bc961da063...,5519,True,PASS



 SUCCESS
Section 2 bridge: Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip (23 files)
Core regenerated files:
 - data/01_document_sentence_entities.csv
 - data/03_full_corpus_with_final_themes.csv
 - data/04_explicit_sentence_relations.csv
 - data/06_explicit_edges_aggregated.csv


In [6]:
# Download the complete upstream output and, for snapshot runs, the ready-to-use
# Section 2 companion input ZIP.
from pathlib import Path

output_archive = Path(result["output_archive"])
files_to_offer = [output_archive]
if EFFECTIVE_MODE.startswith("MANUSCRIPT_SNAPSHOT"):
    files_to_offer.append(OUTPUT_ROOT / BRIDGE_ZIP_NAME)

if AUTO_DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files
        for path in files_to_offer:
            files.download(str(path))
    except Exception:
        for path in files_to_offer:
            print("Available at:", path.resolve())
else:
    for path in files_to_offer:
        print("Available at:", path.resolve())

Available at: /mnt/data/repo_release_build/exec_AISKG_01_Framework_UPLOAD_READY/mushroom_kg_upstream_run_v1/Mushroom_KG_Upstream_Reproducibility_Outputs.zip
Available at: /mnt/data/repo_release_build/exec_AISKG_01_Framework_UPLOAD_READY/mushroom_kg_upstream_run_v1/outputs/Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip


## Continue to Section 2

For the exact manuscript workflow, open the canonical Section 2 repository at [AISKG_02_Framework](https://github.com/romenmeitei/AISKG_02_Framework), then use the generated:

`Mushroom_KG_Reproducibility_Inputs_v2_from_upstream.zip`

This bridge contains the regenerated corpus/theme, entity, sentence-level relation, and aggregated-edge tables plus the frozen expert-validation and held-out benchmark inputs required by the already-tested canonical post-extraction pipeline.

### Optional live-refresh checkpoint

`LIVE_REFRESH / RETRIEVE_AND_MODEL` writes `16_live_topic_expert_review_template.csv`. A domain expert must complete its `expert_label`, `include_exclude`, and `comment` fields. In the same Colab session, set `LIVE_PHASE = "APPLY_CURATION_AND_EXTRACT"`, set `LIVE_EXPERT_REVIEW_FILE` to the completed CSV, and rerun the execution cell. Live outputs are dated updates and should not overwrite the frozen manuscript snapshot.